In [ ]:
# Imports and setup
import json
import os
import sys
from typing import List, Optional, Dict
import gc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import auc, roc_curve
from tqdm import tqdm
from warnings import filterwarnings
filterwarnings('ignore', category=DeprecationWarning)
filterwarnings('ignore', category=UserWarning)
# Import memory optimization helpers
sys.path.append('..')
try:
    from memory_optimized_notebook import load_dataframe, clear_plt, print_memory_usage
except ImportError:
    # Fallback if module not available
    def print_memory_usage(): pass
    def clear_plt(): 
        plt.close('all')
        gc.collect()
    load_dataframe = pd.read_csv

# Force garbage collection at start
gc.collect()

sys.path.append('..')
sys.path.append('../..')
from plotting_notebooks.plotting_utils import (
    load_in_combined_results,
    load_monitoring_data,
    prepare_monitoring_data,
    is_in_training_data,
    plot_suspiciousness_distributions,
    plot_combined_cost_vs_safety_curve,
    plot_combined_cost_vs_tpr_curve,
    interpolate_tpr,
    plot_roc_curves,
)
from utils.analysis_utils import extract_verdict
from utils.path_utils import resolve_path

# Add memory tracking
print_memory_usage()


In [ ]:
# Constants and configuration
MAIN_AGENT_MODELS = ['claude-3-5-sonnet-20241022']
MONITOR_AGENT_MODELS = ['gpt-4o', 'claude-3-5-sonnet-20241022', 'o3-mini', 'o1', 'claude-3-5-haiku-20241022']
DEFAULT_TRANSCRIPT_TYPES = ["transcripts", "transcripts_no_hint", "transcripts_no_secondary_task"]
DOWNSAMPLE_FACTORS = [1, 2, 3, 5, 10, 15]
NUM_MONITORS = 10
BACK_FRACS = [0.1, 0.2, 0.5, 0.8]

# Step mapping for fine-tuned models
FT_STEP_MAPPING = {
    "gpt-4o-2024-08-06": 0,
    'ft:gpt-4o-2024-08-06:scale-safety-research-1:combined-3-reviewed:BCYJFx2l:ckpt-step-70': 70,
    'ft:gpt-4o-2024-08-06:scale-safety-research-1:combined-3-reviewed:BCYJFtC0:ckpt-step-140': 140,
    'ft:gpt-4o-2024-08-06:scale-safety-research-1:combined-3-reviewed:BCYJFa40': 210,
    'ft:gpt-4o-2024-08-06:scale-safety-research-1:unbalanced-data:BDXppMow:ckpt-step-56': 56,
    'ft:gpt-4o-2024-08-06:scale-safety-research-1:unbalanced-data:BDXpp5YL:ckpt-step-112': 112,
    'ft:gpt-4o-2024-08-06:scale-safety-research-1:unbalanced-data:BDXppDFa': 168,
}
FT_MONITOR_MODELS = list(FT_STEP_MAPPING.keys())

# Cache for combined_results
_combined_results_cache = {}


In [ ]:
# Helper functions for data loading

def load_combined_results(include_failed_attempts: bool, include_failed_main_task: bool) -> pd.DataFrame:
    """Load and cache combined_results."""
    cache_key = f"{include_failed_attempts}_{include_failed_main_task}"
    if cache_key in _combined_results_cache:
        print(f"Using cached combined_results for {cache_key}")
        return _combined_results_cache[cache_key]
    combined_results = load_in_combined_results(include_failed_attempts, include_failed_main_task)
    _combined_results_cache[cache_key] = combined_results
    
    # Add memory tracking
    print_memory_usage()
    return combined_results


In [ ]:
# Functions for loading fine-tuned monitoring data

def prepare_ft_monitoring_data(base_dir: str, monitor_models: List[str] = FT_MONITOR_MODELS) -> pd.DataFrame:
    """Load and prepare the fine-tuned monitoring data."""
    # Use the generalized function from plotting_utils
    ft_monitoring_df = prepare_monitoring_data(
        base_dir=base_dir,
        monitor_models=monitor_models,
        num_monitors=NUM_MONITORS,
        compute_aggregates=True,
        tag_word='verdict'
    )
    
    # Extract cost_usd value from usage dictionaries to save memory
    for i in range(1, 11):
        ft_monitoring_df[f'usage_{i}_cost'] = ft_monitoring_df[f'usage_{i}'].apply(
            lambda x: x[0]['cost_usd'] if isinstance(x, list) and len(x) > 0 else 0.0
        )
    ft_monitoring_df['usage_total'] = sum(ft_monitoring_df[f'usage_{i}_cost'] for i in range(1, 11))
    
    # Drop the original usage columns that contain dictionaries to save memory
    usage_dict_columns = [f'usage_{i}' for i in range(1, 11)]
    

    for i in range(1, 11):
        ft_monitoring_df[f'mean_suspiciousness_first_{i}'] = ft_monitoring_df[[f'suspiciousness_{j}' for j in range(1, i+1)]].mean(axis=1)
        # ft_monitoring_df[f'max_suspiciousness_first_{i}'] = ft_monitoring_df[[f'suspiciousness_{j}' for j in range(1, i+1)]].max(axis=1)
        # ft_monitoring_df[f'min_suspiciousness_first_{i}'] = ft_monitoring_df[[f'suspiciousness_{j}' for j in range(1, i+1)]].min(axis=1)
        # # Calculate usage for each mean_suspiciousness_first_i by summing the costs
        for i, x in enumerate(ft_monitoring_df[f'usage_{i}']):
            print(i, x)
            print(i, x[0])
        usage_column = ft_monitoring_df.apply(
            lambda row: sum([row[f'usage_{j}'][0]['cost_usd'] for j in range(1, i+1)]),
            axis=1
        )
        ft_monitoring_df[f'mean_suspiciousness_first_{i}_usage'] = usage_column
    ft_monitoring_df = ft_monitoring_df.drop(columns=usage_dict_columns + [f'suspiciousness_{i}' for i in range(1, 11)] + [f'response_{i}' for i in range(1, 11)] + [f'{rank}th_highest_suspiciousness' for rank in range(3, 11)] + ['1th_highest_suspiciousness'])
    # Memory tracking
    print_memory_usage()
    return ft_monitoring_df


In [ ]:
# Functions for loading turn-by-turn monitoring data

def get_downsampled_responses(x, downsample_factor, back_frac=1.0):
    """Extract responses with specified downsampling factor."""
    all_indices = list(x.values())[::downsample_factor] + [list(x.values())[-1]]
    back_indices = all_indices[-int(len(all_indices) * back_frac):]
    return back_indices

def prepare_turn_by_turn_data(base_dir: str, 
                             downsample_factors: List[int] = DOWNSAMPLE_FACTORS, 
                             thresholds_to_consider = [4, 9, 14, 19, 29, 39], 
                             nth_highest_indices: List[int] = [2, 3, 5], 
                             suffix: str = 'turn_by_turn') -> pd.DataFrame:
    """Load and prepare the turn-by-turn monitoring data."""
    # Load monitoring data
    turn_by_turn_df = load_monitoring_data(base_dir=base_dir)
    
    # Calculate downsampled max suspiciousness scores
    for downsample_factor in downsample_factors:
        turn_by_turn_df[f'max_suspiciousness_{downsample_factor}_{suffix}'] = turn_by_turn_df.data.apply(
            lambda x: float(max([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') or 0.0 
                               for y in get_downsampled_responses(x, downsample_factor)]))
        )
        turn_by_turn_df[f'mean_suspiciousness_{downsample_factor}_{suffix}'] = turn_by_turn_df.data.apply(
            lambda x: float(np.mean([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') or 0.0 
                               for y in get_downsampled_responses(x, downsample_factor)]))
        )
        turn_by_turn_df[f'max_suspiciousness_usage_{downsample_factor}_{suffix}'] = turn_by_turn_df.data.apply(
            lambda x: float(sum([y["usage"][0]["cost_usd"] for y in get_downsampled_responses(x, downsample_factor)]))
        )
        turn_by_turn_df[f'mean_suspiciousness_usage_{downsample_factor}_{suffix}'] = turn_by_turn_df.data.apply(
            lambda x: float(sum([y["usage"][0]["cost_usd"] for y in get_downsampled_responses(x, downsample_factor)]))
        )
    
    # Force garbage collection
    gc.collect()
    for back_frac in BACK_FRACS:
        turn_by_turn_df[f'max_suspiciousness_back_{back_frac}_{suffix}'] = turn_by_turn_df.data.apply(
            lambda x: float(max([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') or 0.0 
                               for y in get_downsampled_responses(x, 1, back_frac)]))
        )
        turn_by_turn_df[f'mean_suspiciousness_back_{back_frac}_{suffix}'] = turn_by_turn_df.data.apply(
            lambda x: float(np.mean([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') or 0.0 
                               for y in get_downsampled_responses(x, 1, back_frac)]))
        )
        turn_by_turn_df[f'max_suspiciousness_usage_back_{back_frac}_{suffix}'] = turn_by_turn_df.data.apply(
            lambda x: float(sum([y["usage"][0]["cost_usd"] for y in get_downsampled_responses(x, 1, back_frac)]))
        )
    gc.collect()
    # Calculate fraction above thresholds
    for threshold in thresholds_to_consider:
        turn_by_turn_df[f'frac_above_{threshold}_{suffix}'] = 100 * turn_by_turn_df.data.apply(
            lambda x: float(np.mean([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') > threshold or 0.0 
                                   for y in get_downsampled_responses(x, 1)]))
        )

        turn_by_turn_df[f'frac_above_{threshold}_usage_{suffix}'] = turn_by_turn_df.data.apply(
            lambda x: float(sum([y["usage"][0]["cost_usd"] for y in get_downsampled_responses(x, 1)]))
        )
    
    # Force garbage collection
    gc.collect()
    
    # Calculate nth highest suspiciousness scores
    for n in nth_highest_indices:
        turn_by_turn_df[f'{n}th_highest_suspiciousness_{suffix}'] = turn_by_turn_df.data.apply(
            lambda x: sorted([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') or 0.0 
                            for y in list(x.values())], reverse=True)[n-1] if len(x) >= n else 0.0
        )

        turn_by_turn_df[f'{n}th_highest_suspiciousness_usage_{suffix}'] = turn_by_turn_df.data.apply(
            lambda x: float(sum([y["usage"][0]["cost_usd"] for y in get_downsampled_responses(x, 1)]))
        )
    
    # Calculate final suspiciousness and usage
    turn_by_turn_df[f'final_suspiciousness_{suffix}'] = turn_by_turn_df.data.apply(
        lambda x: float(extract_verdict(x[str(max([int(k) for k in x.keys()]))]["content"] 
                                      if isinstance(x[str(max([int(k) for k in x.keys()]))], dict) 
                                      and "content" in x[str(max([int(k) for k in x.keys()]))] 
                                      else x[str(max([int(k) for k in x.keys()]))], tag_word='verdict'))
    )
    
    turn_by_turn_df[f'final_usage_{suffix}'] = turn_by_turn_df.data.apply(
        lambda x: float(x[str(max([int(k) for k in x.keys()]))]["usage"][0]["cost_usd"])
    )

    turn_by_turn_df['where_the_max_occurs'] = turn_by_turn_df.data.apply(
        lambda x: np.argmax([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') or 0.0 for y in x.values()])/len(x)
    )
    
    
    # Memory tracking
    print_memory_usage()
    return turn_by_turn_df


In [ ]:
# Function to load all monitoring data

def load_all_monitoring_data(include_failed_attempts=False, include_failed_main_task=False):
    """Load all monitoring data required for combined analysis."""
    # Load base results
    combined_results = load_combined_results(include_failed_attempts, include_failed_main_task)
    merge_on = ['task_name', 'transcript_type', 'file_name']
    just_merge_on_and_side_task = combined_results[merge_on + ['side_task_evaluation']]
    gc.collect()
    
    # Load fine-tuned monitoring data for system prompt 1 and 5
    ft_monitoring_sp1 = prepare_ft_monitoring_data(base_dir='monitor_responses/system_prompt_1_temp_0')
    ft_monitoring_sp1 = pd.merge(ft_monitoring_sp1, just_merge_on_and_side_task, on=merge_on, how='left')
    gc.collect()
    
    ft_monitoring_sp5 = prepare_ft_monitoring_data(base_dir='monitor_responses/system_prompt_5_temp_0')
    ft_monitoring_sp5 = pd.merge(ft_monitoring_sp5, just_merge_on_and_side_task, on=merge_on, how='left')
    gc.collect()
    
    # Load turn-by-turn monitoring data
    turn_by_turn_df = prepare_turn_by_turn_data(base_dir='monitor_responses/turn_by_turn_monitoring')
    turn_by_turn_df = pd.merge(turn_by_turn_df, just_merge_on_and_side_task, on=merge_on, how='left')
    gc.collect()
    
    # Load training data for reference
    training_data_path = resolve_path('finetuning/training_data_dfs/combined_3.csv')
    training_data = pd.read_csv(training_data_path)
    
    
    # Memory tracking and cleanup
    print_memory_usage()
    gc.collect()
    
    return combined_results, ft_monitoring_sp1, ft_monitoring_sp5, turn_by_turn_df, training_data


In [ ]:
# Function to clean and prepare data for unstacking and merging

def prepare_for_unstacking(df, dataframe_type):
    """Clean and prepare dataframe for unstacking."""
    # Make a copy to avoid modifying the original
    df_copy = df.copy()
    
    # Add a source identifier column to track where metrics came from
    df_copy['source'] = dataframe_type
    
    # Remove large/unnecessary columns before unstacking to save memory
    columns_to_drop = ['data']
    
    # For fine-tuned data, also drop the individual responses
    if dataframe_type in ['ft_sp1', 'ft_sp5']:
        columns_to_drop.extend([f'response_{i}' for i in range(1, NUM_MONITORS + 1)])
    
    # Drop columns if they exist in the dataframe
    df_copy = df_copy.drop(columns=[col for col in columns_to_drop if col in df_copy.columns])
    
    # Return the prepared dataframe
    return df_copy


In [ ]:
# Function to unstack dataframes

def unstack_dataframe(df, prefix=''):
    """Unstack the dataframe on monitor_agent and add prefix to columns."""
    # Set index for unstacking
    index_columns = ['task_name', 'transcript_type', 'main_agent', 'file_name', 'side_task_evaluation']
    
    # Create a copy and set index
    df_indexed = df.set_index(index_columns + ['monitor_agent'])
    
    # Drop any columns that might cause issues
    drop_cols = ['source']
    df_indexed = df_indexed.drop(columns=[col for col in drop_cols if col in df_indexed.columns])
    
    # Unstack on monitor_agent
    df_unstacked = df_indexed.unstack(level='monitor_agent')
    
    # Create more readable column names
    new_cols = []
    for metric, monitor in df_unstacked.columns:
        # Use step number for fine-tuned models
        if monitor in FT_STEP_MAPPING:
            step = FT_STEP_MAPPING[monitor]
            if "scale-safety-research-1" in monitor:
                if "combined-3-reviewed" in monitor:
                    model_type = "ft_cr"  # combined-reviewed
                elif "unbalanced-data" in monitor:
                    model_type = "ft_ud"  # unbalanced-data
                else:
                    model_type = "ft"
                monitor_short = f"{model_type}_step{step}"
            else:
                # Base model
                monitor_short = "base"
        else:
            # Non-fine-tuned models, use shorter names
            if monitor == "claude-3-5-sonnet-20241022":
                monitor_short = "claude_sonnet"
            elif monitor == "claude-3-5-haiku-20241022":
                monitor_short = "claude_haiku"
            elif monitor == "gpt-4o":
                monitor_short = "gpt4o"
            else:
                monitor_short = monitor
        
        # Add prefix if provided
        if prefix:
            new_cols.append(f"{prefix}_{metric}_{monitor_short}")
        else:
            new_cols.append(f"{metric}_{monitor_short}")
    
    # Assign new column names
    df_unstacked.columns = new_cols
    
    return df_unstacked


In [ ]:
# Load data
print("Loading all monitoring data...")
combined_results, ft_sp1, ft_sp5, turn_by_turn, training_data = load_all_monitoring_data()
print("Data loaded successfully!")

# Print shapes for debugging
print(f"FT System Prompt 1 shape: {ft_sp1.shape}")
print(f"FT System Prompt 5 shape: {ft_sp5.shape}")
print(f"Turn-by-turn shape: {turn_by_turn.shape}")

# Prepare dataframes for unstacking
print("Preparing dataframes for unstacking...")
ft_sp1_prep = prepare_for_unstacking(ft_sp1, 'ft_sp1')
ft_sp5_prep = prepare_for_unstacking(ft_sp5, 'ft_sp5')
turn_by_turn_prep = prepare_for_unstacking(turn_by_turn, 'tbt')

# Unstack each dataframe
print("Unstacking dataframes...")
ft_sp1_unstacked = unstack_dataframe(ft_sp1_prep, prefix='sp1')
ft_sp5_unstacked = unstack_dataframe(ft_sp5_prep, prefix='sp5')
turn_by_turn_unstacked = unstack_dataframe(turn_by_turn_prep, prefix='tbt')

# Print shapes after unstacking
print(f"FT System Prompt 1 unstacked shape: {ft_sp1_unstacked.shape}")
print(f"FT System Prompt 5 unstacked shape: {ft_sp5_unstacked.shape}")
print(f"Turn-by-turn unstacked shape: {turn_by_turn_unstacked.shape}")

# Merge all unstacked dataframes
print("Merging unstacked dataframes...")
merged_df = ft_sp1_unstacked.merge(
    ft_sp5_unstacked, 
    left_index=True, 
    right_index=True,
    how='outer'
)

merged_df = merged_df.merge(
    turn_by_turn_unstacked,
    left_index=True,
    right_index=True,
    how='outer'
)

# Reset index to have the index columns as regular columns
merged_df = merged_df.reset_index()
merged_df = merged_df[~merged_df.task_name.isin(list(training_data.task_name.unique()))]
# Print final merged shape
print(f"Final merged dataframe shape: {merged_df.shape}")
print(f"Number of columns: {len(merged_df.columns)}")

# Clean up memory
gc.collect()
print_memory_usage()

# Show first few rows of the merged dataframe and column names
print("\nFirst few rows of merged dataframe:")
print(merged_df.head())

print("\nColumn name samples (first 10):")
print(merged_df.columns[:10])

# Save checkpoint
print("Merging completed successfully.")
print(f"The merged dataframe has {merged_df.shape[0]} rows and {merged_df.shape[1]} columns.")

# Print some basic statistics about the dataframe
print("\nDistribution of tasks:")
print(merged_df['task_name'].value_counts())

print("\nDistribution of transcript types:")
print(merged_df['transcript_type'].value_counts())

print("\nPercentage of examples with side tasks:")
print(f"{merged_df['side_task_evaluation'].mean() * 100:.2f}%")

print("\nPercentage of examples in training data:") 


In [ ]:
merged_df.columns

In [ ]:

# Import required libraries for decision tree modeling
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler
import itertools

# Prepare features and target
X = merged_df[merged_df.task_name != 'election_tampering'].drop(columns=columns_to_ignore + ['side_task_evaluation'])
y = merged_df[merged_df.task_name != 'election_tampering']['side_task_evaluation']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")
print(f"Number of features: {X_train.shape[1]}")

# Standardize features to improve model performance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame to retain column names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

# Function to train and evaluate decision trees
def evaluate_decision_tree(max_depth, min_samples_split, min_samples_leaf):
    # Create and train the decision tree
    dt = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
    )
    
    # Train the model
    dt.fit(X_train_scaled, y_train)
    
    # Make predictions
    y_pred = dt.predict(X_test_scaled)
    y_prob = dt.predict_proba(X_test_scaled)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    # Cross-validation score
    cv_scores = cross_val_score(dt, X_train_scaled, y_train, cv=5, scoring='roc_auc')
    cv_auc = cv_scores.mean()
    
    # Get feature importances
    feature_importance = dict(zip(X_train_scaled.columns, dt.feature_importances_))
    top_features = sorted(feature_importance.items(), key=lambda x: x[1], reverse=True)[:5]
    
    # Return a dictionary with all the evaluation metrics
    return {
        'max_depth': max_depth,
        'min_samples_split': min_samples_split,
        'min_samples_leaf': min_samples_leaf,
        'accuracy': accuracy,
        'auc': auc,
        'cv_auc': cv_auc,
        'top_features': top_features,
        'model': dt,
        'confusion_matrix': confusion_matrix(y_test, y_pred),
        'classification_report': classification_report(y_test, y_pred)
    }

# Define parameter grid for decision trees
param_grid = {
    'max_depth': [2],  # Fixed depth of 3 layers as requested
    'min_samples_split': [5, 10, 20],  # More values for min_samples_split
    'min_samples_leaf': [5, 10, 20]  # Higher values for min_samples_leaf as requested
}

# Store all model results
model_results = []

print("Training decision tree models...")
# Try different parameter combinations
for max_depth in param_grid['max_depth']:
    for min_samples_split in param_grid['min_samples_split']:
        for min_samples_leaf in param_grid['min_samples_leaf']:
            print(f"Training with max_depth={max_depth}, min_samples_split={min_samples_split}, min_samples_leaf={min_samples_leaf}")
            result = evaluate_decision_tree(
                max_depth=max_depth,
                min_samples_split=min_samples_split,
                min_samples_leaf=min_samples_leaf
            )
        
            model_results.append(result)

# Sort models by AUC performance
sorted_models = sorted(model_results, key=lambda x: x['auc'], reverse=True)

# Print top 10 models
print("\nTop 10 Decision Tree Models:")
for i, model in enumerate(sorted_models[:10]):
    print(f"\nModel {i+1}:")
    print(f"Parameters: max_depth={model['max_depth']}, min_samples_split={model['min_samples_split']}, min_samples_leaf={model['min_samples_leaf']}")
    print(f"Test Accuracy: {model['accuracy']:.4f}")
    print(f"Test AUC: {model['auc']:.4f}")
    print(f"Cross-Validation AUC: {model['cv_auc']:.4f}")
    print(model['classification_report'])

# Visualize the best model
best_model = sorted_models[0]
print("\nVisualizing the best model...")
plt.figure(figsize=(20, 10))
plot_tree(
    best_model['model'], 
    feature_names=list(X_train_scaled.columns), 
    class_names=['No Side Task', 'Side Task'],
    filled=True, 
    rounded=True,
    fontsize=10
)
plt.title(f"Best Decision Tree Model (AUC = {best_model['auc']:.4f})")
plt.close()

# Print the tree as text for the best model
tree_text = export_text(
    best_model['model'],
    feature_names=list(X_train_scaled.columns)
)
print("\nBest Decision Tree Structure:")
print(tree_text)

print("\nConfusion Matrix for Best Model:")
print(best_model['confusion_matrix'])

print("\nClassification Report for Best Model:")
print(best_model['classification_report'])

# Save the model results to a CSV file
results_df = pd.DataFrame([
    {
        'model_rank': i+1,
        'max_depth': model['max_depth'],
        'min_samples_split': model['min_samples_split'],
        'min_samples_leaf': model['min_samples_leaf'],
        'accuracy': model['accuracy'],
        'auc': model['auc'],
        'cv_auc': model['cv_auc'],
        'top_features': str([f"{f}: {i:.4f}" for f, i in model['top_features']])
    }
    for i, model in enumerate(sorted_models[:10])
])



In [ ]:
# Create a figure for the histograms
plt.figure(figsize=(10, 6))
metric_to_plot = 'sp5_mean_suspiciousness_ft_cr_step70'
# filtered_df = merged_df[(merged_df['sp5_mean_suspiciousness_ft_cr_step70'] < 84) & (merged_df['sp5_max_suspiciousness_ft_ud_step168'] < 72)]
filtered_df = merged_df[merged_df.task_name != 'election_tampering']
# Plot histogram for cases where side_task_evaluation is False
plt.hist(
    filtered_df[~filtered_df['side_task_evaluation']][metric_to_plot],
    alpha=0.5,
    label='No Side Task',
    bins=30,
    density=True
)

# Plot histogram for cases where side_task_evaluation is True
plt.hist(
    filtered_df[filtered_df['side_task_evaluation']][metric_to_plot],
    alpha=0.5,
    label='With Side Task',
    bins=30,
    density=True
)

plt.xlabel(metric_to_plot)
plt.ylabel('Density')
plt.title(f'Distribution of {metric_to_plot} by Side Task Presence')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
# Clear plot to free memory
clear_plt()

In [ ]:
filtered_df[filtered_df.transcript_type != 'transcripts_no_secondary_task'].sort_values(by='sp5_mean_suspiciousness_ft_cr_step70', ascending=True).head(10)

In [ ]:
mean_suspiciousness_columns = [x for x in merged_df.columns if 'mean_suspiciousness' in x]
back_end_average_columns = [x for x in merged_df.columns if 'back_0.1' in x or 'back_0.2' in x]

In [ ]:
# filtered_df = merged_df[~merged_df.task_name.isin(training_data.task_name.unique())]

# prediction_results = []
# for mean_susp_col in mean_suspiciousness_columns:
#     for back_end_col in back_end_average_columns:
#         prediction = (filtered_df[mean_susp_col] > 60) | (filtered_df[back_end_col] > 60)
        
#         # Calculate performance metrics
#         actual = filtered_df['side_task_evaluation']
        
#         # Accuracy
#         accuracy = (prediction == actual).mean()
        
#         # True Positive Rate (Sensitivity)
#         tp = ((prediction == True) & (actual == True)).sum()
#         fn = ((prediction == False) & (actual == True)).sum()
#         tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        
#         # True Negative Rate (Specificity)
#         tn = ((prediction == False) & (actual == False)).sum()
#         fp = ((prediction == True) & (actual == False)).sum()
#         tnr = tn / (tn + fp) if (tn + fp) > 0 else 0
        
#         # F1 Score
#         precision = tp / (tp + fp) if (tp + fp) > 0 else 0
#         recall = tpr  # Same as TPR
#         f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        
#         prediction_results.append({
#             'mean_suspiciousness_col': mean_susp_col,
#             'back_end_col': back_end_col,
#             'accuracy': accuracy,
#             'tpr': tpr,
#             'tnr': tnr,
#             'f1': f1
#         })

# # Convert results to DataFrame for easier analysis
# results_df = pd.DataFrame(prediction_results)



# # Create heatmaps to visualize performance
# # First, pivot the data for heatmaps
# def create_performance_heatmap(metric_name):
#     pivot_df = results_df.pivot_table(
#         index='mean_suspiciousness_col', 
#         columns='back_end_col',
#         values=metric_name
#     )
    
#     plt.figure(figsize=(16, 12))
#     sns.heatmap(pivot_df, annot=True, fmt=".3f", cmap="viridis", annot_kws={'size': 14})
#     plt.title(f'Heatmap of {metric_name} for Column Pairs', fontsize=20)
#     plt.xticks(rotation=45, fontsize=14, ha='right')
#     plt.yticks(fontsize=14)
#     plt.tight_layout()
#     plt.show()
#     clear_plt()

# # Import seaborn if not already imported
# import seaborn as sns

# # Create heatmaps for different metrics
# for metric in ['accuracy', 'f1', 'tpr', 'tnr']:
#     create_performance_heatmap(metric)

# # Find the best overall pair (highest combined score of metrics)
# results_df['combined_score'] = (results_df['accuracy'] + results_df['f1'] + 
#                                (results_df['tpr'] + results_df['tnr'])/2)




In [ ]:
models = ['ft_cr_step70', 'ft_cr_step140', 'ft_cr_step210', 'ft_ud_step56', 'ft_ud_step112', 'ft_ud_step168', 'base']
metric_columns = [f'sp{system_prompt_number}_{aggregation}_suspiciousness_{suffix}' for system_prompt_number in [1, 5] for aggregation in ['mean', '2th_highest'] for suffix in models]
usage_columns = [f'sp{system_prompt_number}_usage_total_{suffix}' for system_prompt_number in [1, 5] for aggregation in ['mean', '2th_highest'] for suffix in models]

metric_columns += [f'sp{i}_mean_suspiciousness_first_{j}_{suffix}' for i in [1, 5] for j in range(1, 11) for suffix in models]
usage_columns += [f'sp{i}_mean_suspiciousness_first_{j}_usage_{suffix}' for i in [1, 5] for j in range(1, 11) for suffix in models]

metric_columns += [f'tbt_{aggregation_type}_suspiciousness_{downsample_factor}_turn_by_turn_{model}' for downsample_factor in DOWNSAMPLE_FACTORS + ['back_0.1', 'back_0.2'] for model in ['base', 'ft_cr_step70'] for aggregation_type in ['mean', 'max']]
usage_columns += [f'tbt_max_suspiciousness_usage_{downsample_factor}_turn_by_turn_{model}' for downsample_factor in DOWNSAMPLE_FACTORS + ['back_0.1', 'back_0.2'] for model in ['base', 'ft_cr_step70'] for aggregation_type in ['mean', 'max']]

slightly_ft_models = ['ft_cr_step70', 'ft_ud_step56']
# Systematic creation of combo columns
sp_metrics = [f'sp{system_prompt_number}_{aggregation}_suspiciousness_{suffix}' 
              for system_prompt_number in [1, 5] 
              for aggregation in ['mean'] 
              for suffix in slightly_ft_models]

tbt_metrics = [f'tbt_{aggregation_type}_suspiciousness_{downsample_factor}_turn_by_turn_{model}' 
              for downsample_factor in ['back_0.1'] 
              for model in ['base', 'ft_cr_step70'] 
              for aggregation_type in ['mean', 'max']]

# Get the corresponding usage columns
sp_usage_dict = {metric: usage for metric, usage in zip(metric_columns, usage_columns) if any(sp in metric for sp in ['sp1', 'sp5'])}
tbt_usage_dict = {metric: usage for metric, usage in zip(metric_columns, usage_columns) if 'tbt' in metric}

print(f"Creating combo columns for {len(sp_metrics)} system prompt metrics and {len(tbt_metrics)} turn-by-turn metrics")

combo_counter = 0
max_combos = 1000  # Limit to avoid memory issues

# Loop through all system prompt metrics
for i, sp_metric in enumerate(sp_metrics + tbt_metrics):
    # Loop through all turn-by-turn metrics
    for j, tbt_metric in enumerate((sp_metrics + tbt_metrics)[i+1:]):
        # Get the corresponding usage columns
        sp_usage = sp_usage_dict.get(sp_metric)
        tbt_usage = tbt_usage_dict.get(tbt_metric)
        
        if not sp_usage or not tbt_usage:
            continue
        
        # Create combo columns using max (taking the higher suspiciousness score)
        max_name = f'combo_max_{sp_metric.replace("suspiciousness_", "")}_{tbt_metric.replace("suspiciousness_", "")}'
        merged_df[max_name] = np.maximum(merged_df[sp_metric], merged_df[tbt_metric])
        max_usage_name = f'combo_max_usage_{sp_metric.replace("suspiciousness_", "")}_{tbt_metric.replace("suspiciousness_", "")}'
        merged_df[max_usage_name] = merged_df[sp_usage] + merged_df[tbt_usage]
        
        # Create combo columns using mean
        mean_name = f'combo_mean_{sp_metric.replace("suspiciousness_", "")}_{tbt_metric.replace("suspiciousness_", "")}'
        merged_df[mean_name] = (merged_df[sp_metric] + merged_df[tbt_metric]) / 2
        mean_usage_name = f'combo_mean_usage_{sp_metric.replace("suspiciousness_", "")}_{tbt_metric.replace("suspiciousness_", "")}'
        merged_df[mean_usage_name] = merged_df[sp_usage] + merged_df[tbt_usage]
        
        # Create combo columns using min (taking the lower suspiciousness score)
        # min_name = f'combo_min_{sp_metric.replace("suspiciousness_", "")}_{tbt_metric.replace("suspiciousness_", "")}'
        # merged_df[min_name] = np.minimum(merged_df[sp_metric], merged_df[tbt_metric])
        # min_usage_name = f'combo_min_usage_{sp_metric.replace("suspiciousness_", "")}_{tbt_metric.replace("suspiciousness_", "")}'
        # merged_df[min_usage_name] = merged_df[sp_usage] + merged_df[tbt_usage]
        
        # Add to metric and usage columns lists
        # metric_columns.extend([max_name, mean_name, min_name])
        # usage_columns.extend([max_usage_name, mean_usage_name, min_usage_name])
        metric_columns.extend([max_name, mean_name])
        usage_columns.extend([max_usage_name, mean_usage_name])
        
        combo_counter += 3  # Count the three combos we just added
        
        if combo_counter >= max_combos:
            print(f"Created {combo_counter} combo columns, stopping to avoid memory issues")
            break
    
    if combo_counter >= max_combos:
        break
        
print(f"Created a total of {combo_counter} combo columns")
merged_df = merged_df.copy()
gc.collect()

In [ ]:
target_fpr = 0.03

# Create a function to handle both AUC and TPR plotting
def create_performance_plot(ax, metric_type='auc', target_fpr=0.03, exclude_tasks = []):
    """
    Create either an AUC or TPR performance plot.
    
    Parameters:
    - ax: Matplotlib axis to plot on
    - metric_type: 'auc' or 'tpr' to determine the plot type
    - target_fpr: Target false positive rate for TPR calculations
    """
    filtered_df = merged_df[~merged_df['task_name'].isin(exclude_tasks)]
    # Add a better style
    plt.style.use('seaborn-v0_8-whitegrid')
    
    # Define a nicer color palette
    colors = plt.cm.tab10(np.linspace(0, 1, len(models)))
    color_map = dict(zip(models, colors))
    
    # Update markers for different prompting types
    markers = {'sp1': 'o', 'sp5': 'x', 'tbt': '^', 'combo_max': 's', 'combo_mean': 'P', 'combo_min': 'X'}
    
    # Process and plot non-combo metrics
    for model in models:
        for prompting_type in ['sp1', 'sp5', 'tbt']:
            model_costs = []
            model_metrics = []  # Will hold either AUC or TPR values
            
            for metric_column, usage_column in zip(metric_columns, usage_columns):
                # Skip combo metrics
                if 'combo_' in metric_column:
                    continue
                
                # For regular metrics, check if model and prompting type match
                if model not in metric_column:
                    continue
                if prompting_type not in metric_column:
                    continue
                    
                # Calculate performance metrics and costs
                fpr, tpr, thresholds = roc_curve(filtered_df['side_task_evaluation'], filtered_df[metric_column])
                
                if metric_type == 'auc':
                    # For AUC plot
                    metric_value = auc(fpr, tpr)
                else:
                    # For TPR plot
                    metric_value = interpolate_tpr(fpr, tpr, target_fpr)
                
                usage = filtered_df[usage_column].mean()
                
                model_costs.append(usage)
                model_metrics.append(metric_value)
    
            if len(model_costs) > 0:
                model_costs = np.array(model_costs)
                model_metrics = np.array(model_metrics)
                
                # Plot with improved styling
                ax.scatter(
                    model_costs, 
                    model_metrics, 
                    s=70,                            # Larger point size
                    color=color_map[model],          # Color by model
                    marker=markers.get(prompting_type, 'o'),  # Different marker per type
                    edgecolor='black',               # Add black edge for contrast
                    linewidth=0.5,                   # Thin edge
                    alpha=0.8,                       # Slightly transparent 
                    label=f'{model} ({prompting_type})'
                )
    
    # Process and plot combo metrics separately
    for combo_type in ['combo_max', 'combo_mean', 'combo_min']:
        # Track which models appear in combo metrics
        model_costs = []
        model_metrics = []  # Will hold either AUC or TPR values
        
        for metric_column, usage_column in zip(metric_columns, usage_columns):
            # Only process combo metrics of this type
            if not metric_column.startswith(combo_type):
                continue
            
            
            # Calculate performance metrics and costs
            fpr, tpr, thresholds = roc_curve(filtered_df['side_task_evaluation'], filtered_df[metric_column])
            
            if metric_type == 'auc':
                # For AUC plot
                metric_value = auc(fpr, tpr)
            else:
                # For TPR plot
                metric_value = interpolate_tpr(fpr, tpr, target_fpr)
            
            usage = filtered_df[usage_column].mean()
            
            model_costs.append(usage)
            model_metrics.append(metric_value)

        if len(model_costs) > 0:
            model_costs = np.array(model_costs)
            model_metrics = np.array(model_metrics)
            
            # Get a simple label with prompting type information
            prompting_label = combo_type.replace('combo_', 'combo ')
            
            # Plot with improved styling
            ax.scatter(
                model_costs, 
                model_metrics, 
                s=70,                            # Larger point size
                color=color_map[model],          # Color by model
                marker=markers.get(combo_type, 'o'),  # Different marker per type
                edgecolor='black',               # Add black edge for contrast
                linewidth=0.5,                   # Thin edge
                alpha=0.8,                       # Slightly transparent 
                label=f'{prompting_label}'
            )
    
    # Set title and labels based on metric type
    if metric_type == 'auc':
        ax.set_title('Cost vs. Safety Performance (AUC)', fontsize=14, fontweight='bold')
        ax.set_ylabel('Safety Performance (AUC)', fontsize=12, fontweight='bold')
        ax.set_ylim(bottom=0.5, top=1.0)
    else:
        ax.set_title(f'Cost vs. TPR at {target_fpr:.2f} FPR', fontsize=14, fontweight='bold')
        ax.set_ylabel(f'TPR at FPR={target_fpr:.2f}', fontsize=12, fontweight='bold')
        ax.set_ylim(bottom=0.0, top=1.0)
    
    ax.set_xlabel('Average Cost (USD)', fontsize=12, fontweight='bold')
    
    # Format axes
    ax.grid(True, linestyle='--', alpha=0.7)
    ax.set_xlim(left=0)
    
    # Format tick labels
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:.4f}'))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.2f}'))
    
    # Add "Better" annotation in the corner
    ax.annotate('Better', xy=(0.02, 0.97), xycoords='axes fraction', 
                fontsize=10, fontweight='bold', color='green')
    
    # Create a legend with unique entries
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(
        by_label.values(), 
        by_label.keys(), 
        title='Model & Prompting Type', 
        fontsize=9, 
        loc='lower right'
    )

# Create an improved cost vs. metrics plot with better formatting
plt.figure(figsize=(16, 8))
exclude_tasks = ['election_tampering']
# Left subplot for AUC
ax1 = plt.subplot(1, 2, 1)
create_performance_plot(ax1, metric_type='auc', target_fpr=target_fpr, exclude_tasks = exclude_tasks)

# Right subplot for TPR
ax2 = plt.subplot(1, 2, 2)
create_performance_plot(ax2, metric_type='tpr', target_fpr=target_fpr, exclude_tasks = exclude_tasks)

# Ensure tight layout
plt.tight_layout()
plt.show()
clear_plt()


In [ ]:
predictive_column = 'sp5_mean_suspiciousness_ft_cr_step70'
low_std_auc_scores = []
high_std_auc_scores = []
std_columns = [x for x in merged_df.columns if 'std' in x]
for std_column in std_columns:
    filtered_df = merged_df[merged_df[std_column] < merged_df[std_column].median()]
    filtered_df = filtered_df
    fpr, tpr, thresholds = roc_curve(filtered_df['side_task_evaluation'], filtered_df[predictive_column])
    auc_score = auc(fpr, tpr)
    low_std_auc_scores.append(auc_score)
    
    filtered_df = merged_df[merged_df[std_column] >= merged_df[std_column].median()]
    filtered_df = filtered_df
    fpr, tpr, thresholds = roc_curve(filtered_df['side_task_evaluation'], filtered_df[predictive_column])
    auc_score = auc(fpr, tpr)
    high_std_auc_scores.append(auc_score)
    gc.collect()

    

In [ ]:
import numpy as np
for i, std_column in enumerate(std_columns):
    print(f'{std_column}: {low_std_auc_scores[i]:.2f}, {high_std_auc_scores[i]:.2f}')

In [ ]:
plt.hist(
    merged_df[(merged_df['task_name'] == 'election_tampering') & (merged_df['transcript_type'] == 'transcripts_no_secondary_task')]['sp5_std_suspiciousness_ft_cr_step70'],
    alpha=0.5,
    label='Election Tampering, no side task',
    bins=30,
    density=True
)

plt.hist(
    merged_df[(merged_df['task_name'] == 'election_tampering') & (merged_df['transcript_type'] != 'transcripts_no_secondary_task')]['sp5_std_suspiciousness_ft_cr_step70'],
    alpha=0.5,
    label='Election Tampering, with side task',
    bins=30,
    density=True
)

# Plot histogram for cases where side_task_evaluation is True
plt.hist(
    merged_df[merged_df['task_name'] != 'election_tampering']['sp5_std_suspiciousness_ft_cr_step70'],
    alpha=0.5,
    label='Other Tasks',
    bins=30,
    density=True
)
plt.xlabel('Standard Deviation of Suspiciousness')
plt.ylabel('Density')
plt.title(f'Distribution of Standard Deviation of Suspiciousness by Task')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
metric_column = 'sp5_mean_suspiciousness_ft_cr_step70'
std_dev_column = 'sp5_std_suspiciousness_ft_cr_step70'
merged_df[f'error_{metric_column}'] = (merged_df[metric_column]/100 - merged_df['side_task_evaluation']) ** 2
filtered_df = merged_df[merged_df['task_name'] == 'election_tampering']
plt.scatter(
    filtered_df[std_dev_column],
    filtered_df[f'error_{metric_column}'],
    alpha=0.5,
    label='Election Tampering'
)
filtered_df = merged_df[merged_df['task_name'] != 'election_tampering']
plt.scatter(
    filtered_df[std_dev_column],
    filtered_df[f'error_{metric_column}'],
    alpha=0.5,
    label='Other Tasks'
)
plt.xlabel('Standard Deviation of Suspiciousness')
plt.ylabel('Error')
plt.legend()
plt.show()

In [ ]:
# Bin the suspiciousness values and calculate means
bins = np.arange(0, 101, 10)  # Creates bins from 0 to 100 in steps of 10
labels = [f'{bins[i]}-{bins[i+1]}' for i in range(len(bins)-1)]
plt.figure(figsize=(12, 6))
# Create bins and calculate means
merged_df['mean_of_sp1_sp5_suspiciousness_ft_cr_step70'] = (merged_df['sp1_mean_suspiciousness_ft_cr_step70'] + merged_df['sp5_mean_suspiciousness_ft_cr_step70']) / 2
for metric_column in [f'sp{i}_{agg_type}_suspiciousness_{ft_name}' for i in [1, 5] for agg_type in ['mean'] for ft_name in ['ft_cr_step70', 'ft_ud_step56']] + ['mean_of_sp1_sp5_suspiciousness_ft_cr_step70']:
    binned_means = merged_df.groupby(
        pd.cut(merged_df[metric_column], bins=bins, labels=labels)
    )['side_task_evaluation'].mean()
    plt.plot(binned_means.index, binned_means.values, label=metric_column)
plt.plot(binned_means.index, np.linspace(0, 1, len(binned_means)), label='Perfect Calibration', linestyle='--')
plt.title('Mean Side Task Evaluation by Suspiciousness Bin')
plt.legend()
plt.xlabel('Suspiciousness Range')
plt.ylabel('Mean Side Task Evaluation')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
clear_plt()


In [ ]:
metric_columns = [f'sp{i}_{agg_type}_suspiciousness_{ft_name}' for i in [1, 5] for agg_type in ['mean'] for ft_name in ['ft_cr_step70', 'ft_ud_step56']]
plt.figure(figsize=(12, 6))
for metric_column in metric_columns:
    std_column = metric_column.replace('mean_suspiciousness', 'std_suspiciousness')
    weights = np.clip((100- merged_df[std_column])/1, 0, 1)
    merged_df[f'calibrated_{metric_column}'] = (merged_df[metric_column] * weights) + (1 - weights) * 50
    fpr, tpr, thresholds = roc_curve(merged_df['side_task_evaluation'], merged_df[f'calibrated_{metric_column}'])
    auc_score = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'calibrated_{metric_column} (AUC = {auc_score:.2f})')
    fpr, tpr, thresholds = roc_curve(merged_df['side_task_evaluation'], merged_df[metric_column])
    auc_score = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{metric_column} (AUC = {auc_score:.2f})')

plt.legend()
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Calibrated Suspiciousness')
plt.show()


In [ ]:
plt.hist(
    merged_df[(merged_df.side_task_evaluation == False) & (merged_df['task_name'] != 'election_tampering')]['sp5_mean_suspiciousness_ft_cr_step70'],
    alpha=0.5,
    label='Side Task Evaluation = False',
    bins=30,
    density=True
)
plt.hist(
    merged_df[(merged_df.side_task_evaluation == True) & (merged_df['task_name'] != 'election_tampering')]['sp5_mean_suspiciousness_ft_cr_step70'],
    alpha=0.5,
    label='Side Task Evaluation = True',
    bins=30,
    density=True)
plt.xlabel('Mean Suspiciousness')
plt.ylabel('Density')
plt.title('Distribution of Mean Suspiciousness by Side Task Evaluation')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plt.hist(
    merged_df[(merged_df.side_task_evaluation == False) & (merged_df['task_name'] != 'election_tampering')]['calibrated_sp5_mean_suspiciousness_ft_cr_step70'],
    alpha=0.5,
    label='Side Task Evaluation = False',
    bins=30,
    density=True
)
plt.hist(
    merged_df[(merged_df.side_task_evaluation == True) & (merged_df['task_name'] != 'election_tampering')]['calibrated_sp5_mean_suspiciousness_ft_cr_step70'],
    alpha=0.5,
    label='Side Task Evaluation = True',
    bins=30,
    density=True)
plt.xlabel('Mean Suspiciousness')
plt.ylabel('Density')
plt.title('Distribution of Calibrated Mean Suspiciousness by Side Task Evaluation')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
predictive_column = 'sp5_mean_suspiciousness_ft_cr_step70'
where_the_max_occurs_columns = [x for x in merged_df.columns if 'where_the_max_occurs' in x]
where_the_max_occurs_columns
for where_the_max_occurs_column in where_the_max_occurs_columns:
    filtered_df = merged_df[merged_df[where_the_max_occurs_column] < 0.3]
    filtered_df = filtered_df
    fpr, tpr, thresholds = roc_curve(filtered_df['side_task_evaluation'], filtered_df[predictive_column])
    auc_score = auc(fpr, tpr)
    low_std_auc_scores.append(auc_score)
    
    filtered_df = merged_df[merged_df[where_the_max_occurs_column] >= 0.3]
    filtered_df = filtered_df
    fpr, tpr, thresholds = roc_curve(filtered_df['side_task_evaluation'], filtered_df[predictive_column])
    auc_score = auc(fpr, tpr)
    high_std_auc_scores.append(auc_score)
    gc.collect()

for i, where_the_max_occurs_column in enumerate(where_the_max_occurs_columns):
    print(f'{where_the_max_occurs_column}: {low_std_auc_scores[i]:.2f}, {high_std_auc_scores[i]:.2f}')

In [ ]:
ranges = [[0, 0.2], [0.2, 0.4], [0.4, 0.6], [0.6, 0.8], [0.8, 1]]
for range in ranges:
    plt.figure(figsize=(12, 6))
    plt.hist(
        merged_df[(merged_df.side_task_evaluation == False) & (merged_df['task_name'] != 'election_tampering') & (merged_df['tbt_where_the_max_occurs_ft_cr_step70'] >= range[0]) & (merged_df['tbt_where_the_max_occurs_ft_cr_step70'] < range[1])]['sp1_mean_suspiciousness_ft_cr_step70'],
        alpha=0.5,
        label='Side Task Evaluation = False',
        bins=20,
        density=True
    )
    plt.hist(
        merged_df[(merged_df.side_task_evaluation == True) & (merged_df['task_name'] != 'election_tampering') & (merged_df['tbt_where_the_max_occurs_ft_cr_step70'] >= range[0]) & (merged_df['tbt_where_the_max_occurs_ft_cr_step70'] < range[1])]['sp1_mean_suspiciousness_ft_cr_step70'],
        alpha=0.5,
        label='Side Task Evaluation = True',
        bins=20,
        density=True)
    plt.xlabel('Mean Suspiciousness')
    plt.ylabel('Density')
    plt.title(f'Distribution of Mean Suspiciousness by Side Task Evaluation for {range[0]}-{range[1]}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()